# DFA-Based Pattern Matching

This notebook covers **Deterministic Finite Automaton (DFA)** based string pattern matching - a technique that preprocesses the pattern to build a state machine enabling O(n) search time with no backtracking.

## Table of Contents
1. [Finite Automaton Basics](#1-finite-automaton-basics)
2. [DFA for Pattern Matching](#2-dfa-for-pattern-matching)
3. [Building the Transition Table](#3-building-the-transition-table)
4. [Searching with DFA](#4-searching-with-dfa)
5. [Implementation](#5-implementation)
6. [Examples](#6-examples)
7. [DFA vs KMP Comparison](#7-dfa-vs-kmp-comparison)

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook builds algorithmic thinking used to reason about performance and correctness in computational biology tools.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Big-O is asymptotic guidance; constant factors still matter in practice.
- Correctness comes before optimization: verify edge cases before performance tuning.
- Data structure choice often dominates speed more than micro-optimizations.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Rabin-Karp Algorithm](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/03_rabin_karp.ipynb) | [Next: Trie (Prefix Tree) →](../../Tier_4_Algorithms_and_Data_Structures/08_Advanced_String_Structures/01_tries.ipynb)

---

---
## 1. Finite Automaton Basics

A **Deterministic Finite Automaton (DFA)** is formally defined as a 5-tuple:

$$\text{DFA} = (Q, \Sigma, \delta, q_0, F)$$

where:
- **Q** = finite set of states
- **Σ** (Sigma) = input alphabet (finite set of symbols)
- **δ** (delta) = transition function: Q × Σ → Q
- **q₀** = start state (q₀ ∈ Q)
- **F** = set of accepting (final) states (F ⊆ Q)

### Key Properties

1. **Deterministic**: For each state and input symbol, there is exactly ONE next state
2. **Complete**: Every state has a transition defined for every symbol in Σ
3. **No ε-transitions**: Only transitions on actual input symbols

### DFA Operation

```
Given input string w = a₁a₂...aₙ:

1. Start at state q₀
2. For each character aᵢ:
   current_state = δ(current_state, aᵢ)
3. Accept if final state ∈ F
```

---
## 2. DFA for Pattern Matching

For pattern matching, we construct a DFA where:

- **State i** means "we have matched the first i characters of the pattern"
- **State 0** = no characters matched (start state)
- **State m** = all m characters matched (accepting state)

### Formal Definition for Pattern P[0..m-1]

$$\text{DFA} = (Q, \Sigma, \delta, 0, \{m\})$$

- **Q** = {0, 1, 2, ..., m} — (m + 1) states
- **Σ** = alphabet of the text
- **q₀** = 0 (no match)
- **F** = {m} (complete match)

### State Meaning

```
Pattern: "ABABC" (m = 5)

State 0: No characters matched     → matched ""
State 1: Matched P[0]              → matched "A"
State 2: Matched P[0..1]           → matched "AB"
State 3: Matched P[0..2]           → matched "ABA"
State 4: Matched P[0..3]           → matched "ABAB"
State 5: Matched P[0..4]           → matched "ABABC" ✓ ACCEPTING
```

### ASCII Art: DFA for Pattern "ABABC"

```
States: 0, 1, 2, 3, 4, 5 (5 = accepting state)

      A       B       A       B       C
→(0)────→(1)────→(2)────→(3)────→(4)────→((5))
   ↑           │       │
   │     B     │   A   │
   └───────────┘       │
         │             │
         └─────────────┘
              B

Legend:
  →(0)   = Start state
  ((5))  = Accepting (double circle)
  ────→  = Transition on matching character
```

### Complete Transition Table

```
┌─────────────────────────────────────────────┐
│ State │  A  │  B  │  C  │ (other)           │
├───────┼─────┼─────┼─────┼───────────────────┤
│   0   │  1  │  0  │  0  │    0              │
│   1   │  1  │  2  │  0  │    0              │
│   2   │  3  │  0  │  0  │    0              │
│   3   │  1  │  4  │  0  │    0              │
│   4   │  3  │  0  │  5  │    0              │
│   5   │  -  │  -  │  -  │    - (accepting)  │
└───────┴─────┴─────┴─────┴───────────────────┘

Reading the table:
- Row = current state
- Column = input character
- Cell value = next state
```

---
## 3. Building the Transition Table

The key insight is computing δ(state, char) for ALL combinations:

### Transition Function Logic

For state `q` and character `c`:

```
δ(q, c) = length of LONGEST prefix of P that is also
          a suffix of (P[0..q-1] + c)
```

In other words:
- We've matched P[0..q-1]
- We see character c
- What's the longest prefix of P that ends with this new state?

### Two Cases

**Case 1: Character matches next pattern character**
```
If c == P[q], then δ(q, c) = q + 1
(We extend the match by one character)
```

**Case 2: Character doesn't match (mismatch)**
```
Find the longest proper prefix of P[0..q-1] + c
that is also a suffix of the pattern.

This is where we use the PREFIX FUNCTION!
```

### Building Transitions Using Prefix Function

```
Pattern: "ABABC"
Prefix function π: [0, 0, 1, 2, 0]

═══════════════════════════════════════════════════════
Example 1: δ(2, 'A')  — state=2 (matched "AB"), char='A'
═══════════════════════════════════════════════════════

  Does 'A' match P[2]? P[2]='A' → YES!
  δ(2, 'A') = 2 + 1 = 3

  Meaning: "AB" + "A" = "ABA" = P[0..2] ✓

═══════════════════════════════════════════════════════
Example 2: δ(2, 'C')  — state=2 (matched "AB"), char='C'
═══════════════════════════════════════════════════════

  Does 'C' match P[2]? P[2]='A' → NO
  
  We need to find: longest prefix of P that is suffix of "ABC"
  
  Method: Use prefix function recursively
  - We had matched "AB" (state 2)
  - Fall back to π[1] = 0 (longest proper prefix-suffix of "A")
  - From state 0, check: does 'C' match P[0]? P[0]='A' → NO
  - δ(2, 'C') = 0

═══════════════════════════════════════════════════════
Example 3: δ(4, 'A')  — state=4 (matched "ABAB"), char='A'
═══════════════════════════════════════════════════════

  Does 'A' match P[4]? P[4]='C' → NO
  
  Fall back using prefix function:
  - π[3] = 2 ("ABAB" has prefix-suffix "AB" of length 2)
  - From state 2, check: does 'A' match P[2]? P[2]='A' → YES!
  - δ(4, 'A') = 2 + 1 = 3

  Meaning: "ABAB" + "A" → keep suffix "ABA" = P[0..2]
```

### Naive vs Optimized Construction

**Naive Approach: O(m³|Σ|)**
```python
for state in range(m + 1):
    for char in alphabet:
        # Try all possible prefix lengths
        for k in range(min(m, state + 1), 0, -1):
            if pattern[:k] == (pattern[:state] + char)[-k:]:
                delta[state][char] = k
                break
```

**Optimized Approach: O(m|Σ|)** using prefix function
```python
for state in range(m + 1):
    for char in alphabet:
        # Use prefix function for O(1) amortized lookup
        k = state
        while k > 0 and pattern[k] != char:
            k = prefix[k - 1]
        if pattern[k] == char:
            k += 1
        delta[state][char] = k
```

---
## 4. Searching with DFA

Once the DFA is built, searching is simple and fast:

```
DFA_SEARCH(text, dfa):
    state = 0
    for i = 0 to len(text) - 1:
        state = dfa[state][text[i]]
        if state == m:           # Accepting state
            report match at position (i - m + 1)
```

### Search Example

```
Pattern: "ABABC" (m = 5)
Text:    "ABABABC"

Step-by-step state transitions:

Position:  0    1    2    3    4    5    6
Character: A    B    A    B    A    B    C
           ↓    ↓    ↓    ↓    ↓    ↓    ↓
State:  0→ 1 → 2 → 3 → 4 → 3 → 4 → 5
                          │         ↑
                          │    FOUND! ✓
                          │
                    Mismatch at state 4:
                    Expected 'C', got 'A'
                    δ(4,'A') = 3 (fall back)

Match found at position: 6 - 5 + 1 = 2

Verification: text[2:7] = "ABABC" ✓
```

### Finding All Occurrences

For overlapping matches, continue searching after finding a match:

```
Pattern: "ABA"
Text:    "ABABABA"

Char:  A  B  A  B  A  B  A
       ↓  ↓  ↓  ↓  ↓  ↓  ↓
State: 0→1→2→3→2→3→2→3
             ↑     ↑     ↑
          Match  Match  Match
          pos=0  pos=2  pos=4

Note: After reaching accepting state 3,
      we transition using dfa[3][next_char]
      to continue finding overlapping matches.
```

---
## 5. Implementation

In [ ]:
def compute_prefix_function(pattern: str) -> list:
    """
    Compute the prefix function (failure function) for the pattern.
    
    The prefix function π[i] is the length of the longest proper prefix
    of pattern[0..i] that is also a suffix of pattern[0..i].
    
    Parameters
    ----------
    pattern : str
        The pattern to compute the prefix function for.
    
    Returns
    -------
    list
        Prefix function array of length m.
    
    Time Complexity: O(m) where m = len(pattern)
    Space Complexity: O(m)
    
    Example
    -------
    >>> compute_prefix_function("ABABC")
    [0, 0, 1, 2, 0]
    """
    m = len(pattern)
    if m == 0:
        return []
    
    prefix = [0] * m
    k = 0  # length of previous longest prefix-suffix
    
    for i in range(1, m):
        # Fall back using prefix function until we find a match or reach start
        while k > 0 and pattern[i] != pattern[k]:
            k = prefix[k - 1]
        
        if pattern[i] == pattern[k]:
            k += 1
        
        prefix[i] = k
    
    return prefix


# Test the prefix function
print("Prefix function examples:")
for pattern in ["ABABC", "AAAA", "ABCABC", "ATTCTGATTT"]:
    print(f"  π('{pattern}') = {compute_prefix_function(pattern)}")

In [ ]:
def build_dfa(pattern: str, alphabet: str) -> dict:
    """
    Build a DFA transition table for pattern matching.
    
    Constructs a deterministic finite automaton where state i represents
    having matched the first i characters of the pattern.
    
    Parameters
    ----------
    pattern : str
        The pattern to search for.
    alphabet : str
        String containing all possible characters in the text.
    
    Returns
    -------
    dict
        Nested dictionary: dfa[state][char] -> next_state
        States range from 0 to m (inclusive).
    
    Time Complexity: O(m × |Σ|) where m = len(pattern), |Σ| = len(alphabet)
    Space Complexity: O(m × |Σ|)
    
    Example
    -------
    >>> dfa = build_dfa("AB", "ABC")
    >>> dfa[0]['A']  # From state 0, reading 'A'
    1
    >>> dfa[1]['B']  # From state 1, reading 'B'
    2  # Accepting state
    """
    m = len(pattern)
    
    # Handle empty pattern
    if m == 0:
        return {0: {c: 0 for c in alphabet}}
    
    # Compute prefix function for the pattern
    prefix = compute_prefix_function(pattern)
    
    # Build transition table for states 0 to m
    dfa = {}
    
    for state in range(m + 1):
        dfa[state] = {}
        
        for char in alphabet:
            if state < m and char == pattern[state]:
                # Character matches: advance to next state
                dfa[state][char] = state + 1
            else:
                # Mismatch: use prefix function to find fallback state
                # Find longest prefix of pattern that is suffix of (pattern[0..state-1] + char)
                k = state
                while k > 0:
                    k = prefix[k - 1]
                    if k < m and char == pattern[k]:
                        k += 1
                        break
                    if k == 0:
                        if char == pattern[0]:
                            k = 1
                        break
                dfa[state][char] = k
    
    return dfa


# Test DFA construction
test_dfa = build_dfa("AB", "ABC")
print("DFA for pattern 'AB':")
for state in test_dfa:
    print(f"  State {state}: {test_dfa[state]}")

In [ ]:
def build_dfa_via_suffix_check(pattern: str, alphabet: str) -> dict:
    """
    Build DFA using direct suffix matching (alternative implementation).
    
    For each state and character, computes the longest prefix of pattern
    that matches a suffix of (pattern[0:state] + char).
    
    Parameters
    ----------
    pattern : str
        The pattern to search for.
    alphabet : str
        String containing all possible characters in the text.
    
    Returns
    -------
    dict
        Nested dictionary: dfa[state][char] -> next_state
    
    Time Complexity: O(m² × |Σ|) - less efficient but more intuitive
    Space Complexity: O(m × |Σ|)
    """
    m = len(pattern)
    dfa = {}
    
    for state in range(m + 1):
        dfa[state] = {}
        current_matched = pattern[:state]  # What we've matched so far
        
        for char in alphabet:
            # After reading char, what's the longest prefix of pattern
            # that is a suffix of (current_matched + char)?
            test_string = current_matched + char
            
            # Try all possible prefix lengths, from longest to shortest
            for k in range(min(m, len(test_string)), -1, -1):
                if test_string.endswith(pattern[:k]):
                    dfa[state][char] = k
                    break
    
    return dfa


# Verify both implementations give same results
pattern = "ABABC"
alphabet = "ABC"

dfa1 = build_dfa(pattern, alphabet)
dfa2 = build_dfa_via_suffix_check(pattern, alphabet)

print(f"Both implementations match: {dfa1 == dfa2}")

In [ ]:
def dfa_search(text: str, pattern: str, alphabet: str = None) -> list:
    """
    Search for all occurrences of pattern in text using DFA.
    
    Builds a DFA from the pattern and uses it to scan the text in a single
    pass. Each character is processed exactly once with O(1) state lookup.
    
    Parameters
    ----------
    text : str
        The text to search in.
    pattern : str
        The pattern to search for.
    alphabet : str, optional
        Characters that can appear in text. If None, derived from text.
    
    Returns
    -------
    list
        List of starting positions (0-indexed) where pattern occurs.
    
    Time Complexity:
        - Preprocessing: O(m × |Σ|) to build DFA
        - Search: O(n) - single pass through text
        - Total: O(m × |Σ| + n)
    
    Space Complexity: O(m × |Σ|) for the DFA table
    
    Example
    -------
    >>> dfa_search("ABABABC", "ABABC")
    [2]
    >>> dfa_search("ABABABA", "ABA")
    [0, 2, 4]
    """
    if len(pattern) == 0:
        return list(range(len(text) + 1))  # Empty pattern matches everywhere
    
    if len(pattern) > len(text):
        return []
    
    # Derive alphabet from text if not provided
    if alphabet is None:
        alphabet = ''.join(sorted(set(text)))
    
    # Build the DFA
    m = len(pattern)
    dfa = build_dfa(pattern, alphabet)
    
    # Search using DFA
    matches = []
    state = 0
    
    for i, char in enumerate(text):
        # Transition to next state
        state = dfa[state].get(char, 0)  # Default to 0 for unknown chars
        
        # Check for match (reached accepting state)
        if state == m:
            match_pos = i - m + 1
            matches.append(match_pos)
            # Continue searching for overlapping matches
            # State stays at m, next transition will use dfa[m][next_char]
    
    return matches


# Test the search function
print("Search tests:")
print(f"  'ABABC' in 'ABABABC': {dfa_search('ABABABC', 'ABABC')}")
print(f"  'ABA' in 'ABABABA':   {dfa_search('ABABABA', 'ABA')}")
print(f"  'ABC' in 'ABCABCABC': {dfa_search('ABCABCABC', 'ABC')}")

In [ ]:
def dfa_search_verbose(text: str, pattern: str, alphabet: str = None) -> list:
    """
    DFA search with detailed output showing state transitions.
    
    Same as dfa_search but prints the transition table and
    step-by-step state changes during the search.
    
    Parameters
    ----------
    text : str
        The text to search in.
    pattern : str
        The pattern to search for.
    alphabet : str, optional
        Characters that can appear in text.
    
    Returns
    -------
    list
        List of starting positions where pattern occurs.
    """
    if len(pattern) == 0 or len(pattern) > len(text):
        return [] if len(pattern) > len(text) else list(range(len(text) + 1))
    
    if alphabet is None:
        alphabet = ''.join(sorted(set(text + pattern)))
    
    m = len(pattern)
    dfa = build_dfa(pattern, alphabet)
    
    # Print transition table
    print(f"Pattern: '{pattern}'")
    print(f"Text:    '{text}'")
    print(f"\nTransition Table (DFA):")
    
    # Header
    header = "State │ " + " │ ".join(f"{c:^3}" for c in alphabet)
    separator = "──────┼" + "┼".join("─────" for _ in alphabet)
    print(f"┌{'─' * (len(header) + 1)}┐")
    print(f"│ {header} │")
    print(f"├{separator}┤")
    
    for state in range(m + 1):
        row = f"  {state:^3} │ "
        row += " │ ".join(f"{dfa[state].get(c, 0):^3}" for c in alphabet)
        accepting = " ← accepting" if state == m else ""
        print(f"│{row} │{accepting}")
    print(f"└{'─' * (len(header) + 1)}┘")
    
    # Search with trace
    print(f"\nSearch trace:")
    print("─" * 50)
    
    matches = []
    state = 0
    
    for i, char in enumerate(text):
        prev_state = state
        state = dfa[state].get(char, 0)
        
        match_indicator = ""
        if state == m:
            match_pos = i - m + 1
            matches.append(match_pos)
            match_indicator = f"  ✓ MATCH at position {match_pos}"
        
        print(f"  i={i:2}, char='{char}': state {prev_state} → {state}{match_indicator}")
    
    print("─" * 50)
    print(f"\nMatches found: {matches}")
    
    return matches

---
## 6. Examples

### Example 1: Basic Pattern Matching

In [ ]:
# Simple example with step-by-step trace
dfa_search_verbose("ABABABC", "ABABC")

### Example 2: Overlapping Matches

In [ ]:
# Finding overlapping occurrences
dfa_search_verbose("ABABABA", "ABA")

### Example 3: DNA Sequence Matching

In [ ]:
# DNA pattern matching (Σ = {A, T, G, C})
dna_text = "AATGCCGTATTCTATTCTGATTCTGATTAGT"
dna_pattern = "ATTCT"
dna_alphabet = "ATGC"

print("DNA Sequence Pattern Matching")
print("=" * 50)
dfa_search_verbose(dna_text, dna_pattern, dna_alphabet)

In [ ]:
# Longer DNA pattern from reference
dna_text = "AATGCCGTATTCTATTCTGATTTCTGAATTCTGATTTTTAGT"
dna_pattern = "ATTCTGATTT"
dna_alphabet = "ATGC"

print("Longer DNA Pattern Matching")
print("=" * 50)
matches = dfa_search(dna_text, dna_pattern, dna_alphabet)
print(f"Pattern: '{dna_pattern}'")
print(f"Text:    '{dna_text}'")
print(f"\nMatches found at positions: {matches}")

# Verify matches
print("\nVerification:")
for pos in matches:
    found = dna_text[pos:pos + len(dna_pattern)]
    print(f"  Position {pos}: '{found}' == '{dna_pattern}' → {found == dna_pattern}")

### Example 4: Building DFA Step by Step

In [ ]:
def visualize_dfa_construction(pattern: str, alphabet: str):
    """
    Visualize the DFA construction process step by step.
    """
    print(f"Building DFA for pattern: '{pattern}'")
    print(f"Alphabet: {{{', '.join(alphabet)}}}")
    print("=" * 60)
    
    # Compute prefix function
    prefix = compute_prefix_function(pattern)
    print(f"\nStep 1: Compute prefix function")
    print(f"  Pattern:  {' '.join(pattern)}")
    print(f"  Index:    {' '.join(str(i) for i in range(len(pattern)))}")
    print(f"  π:        {' '.join(str(p) for p in prefix)}")
    
    print(f"\nStep 2: Build transitions for each state")
    print("-" * 60)
    
    m = len(pattern)
    dfa = {}
    
    for state in range(m + 1):
        dfa[state] = {}
        matched = pattern[:state] if state > 0 else "ε"
        print(f"\nState {state} (matched: '{matched}'):")
        
        for char in alphabet:
            if state < m and char == pattern[state]:
                next_state = state + 1
                reason = f"'{char}' matches P[{state}]='{pattern[state]}' → advance"
            else:
                # Compute using suffix check for clarity
                test_string = pattern[:state] + char
                for k in range(min(m, len(test_string)), -1, -1):
                    if test_string.endswith(pattern[:k]):
                        next_state = k
                        break
                if state < m:
                    reason = f"'{char}' ≠ P[{state}]='{pattern[state]}' → fallback"
                else:
                    reason = f"accepting state → restart matching"
            
            dfa[state][char] = next_state
            print(f"    δ({state}, '{char}') = {next_state}  ({reason})")
    
    return dfa


# Demonstrate construction
dfa = visualize_dfa_construction("ABABC", "ABC")

### Example 5: Pretty-Printed Transition Table

In [ ]:
def print_transition_table(pattern: str, alphabet: str):
    """
    Print a nicely formatted transition table for the DFA.
    """
    dfa = build_dfa(pattern, alphabet)
    m = len(pattern)
    
    # Calculate column width
    col_width = 5
    
    # Build table
    print(f"\nDFA Transition Table for Pattern: '{pattern}'")
    print(f"Alphabet: {{{', '.join(alphabet)}}}")
    print()
    
    # Header
    header = f"{'State':^7}│" + "│".join(f"{c:^{col_width}}" for c in alphabet) + "│ Matched"
    line = "─" * 7 + "┼" + "┼".join("─" * col_width for _ in alphabet) + "┼" + "─" * 10
    
    print("┌" + "─" * 7 + "┬" + "┬".join("─" * col_width for _ in alphabet) + "┬" + "─" * 10 + "┐")
    print("│" + header + " │")
    print("├" + line + "┤")
    
    # Rows
    for state in range(m + 1):
        matched = pattern[:state] if state > 0 else "ε"
        state_str = f"{'→' if state == 0 else ' '}{state:2}{'*' if state == m else ' '}"
        row = f"{state_str:^7}│"
        row += "│".join(f"{dfa[state].get(c, 0):^{col_width}}" for c in alphabet)
        row += f"│ {matched:^8} │"
        print("│" + row)
    
    print("└" + "─" * 7 + "┴" + "┴".join("─" * col_width for _ in alphabet) + "┴" + "─" * 10 + "┘")
    print("\nLegend: → = start state, * = accepting state, ε = empty string")


# Show transition tables for different patterns
print_transition_table("ABABC", "ABC")
print()
print_transition_table("AABA", "AB")
print()
print_transition_table("ATTCT", "ATGC")

---
## 7. DFA vs KMP Comparison

### Complexity Comparison

| Aspect | DFA | KMP |
|--------|-----|-----|
| **Preprocessing Time** | O(m × \|Σ\|) | O(m) |
| **Preprocessing Space** | O(m × \|Σ\|) | O(m) |
| **Search Time** | O(n) | O(n) |
| **Search Space** | O(1) | O(1) |
| **Per-character work** | 1 table lookup | Up to m prefix lookups (amortized O(1)) |

### Trade-offs

```
Choose DFA when:
├── Alphabet is SMALL (DNA: |Σ|=4, binary: |Σ|=2)
├── Searching the SAME pattern in MANY texts
├── Need FASTEST possible search (no conditionals per char)
└── Memory is not a constraint

Choose KMP when:
├── Alphabet is LARGE (Unicode: |Σ|=millions)
├── Pattern changes frequently
├── Memory is constrained
└── One-time search (preprocessing overhead matters)
```

### Why DFA Search is Faster

```
DFA search per character:
    state = dfa[state][char]   ← Single table lookup, O(1)

KMP search per character:
    while j > 0 and text[i] != pattern[j]:   ← Potential loop
        j = prefix[j - 1]
    if text[i] == pattern[j]:
        j += 1

Although KMP is O(n) overall (amortized), each character may
require multiple prefix function lookups. DFA guarantees
exactly ONE operation per character.
```

In [ ]:
import time

def kmp_search(text: str, pattern: str) -> list:
    """KMP search for comparison."""
    if not pattern:
        return list(range(len(text) + 1))
    
    # Build prefix function
    m = len(pattern)
    prefix = compute_prefix_function(pattern)
    
    # Search
    matches = []
    j = 0  # Characters matched in pattern
    
    for i, char in enumerate(text):
        while j > 0 and char != pattern[j]:
            j = prefix[j - 1]
        
        if char == pattern[j]:
            j += 1
        
        if j == m:
            matches.append(i - m + 1)
            j = prefix[j - 1]
    
    return matches


def benchmark_comparison(text: str, pattern: str, alphabet: str, num_runs: int = 100):
    """
    Compare DFA and KMP performance.
    """
    print(f"Benchmark: pattern length = {len(pattern)}, text length = {len(text)}")
    print(f"Alphabet size = {len(alphabet)}")
    print("-" * 50)
    
    # Pre-build DFA (one-time cost)
    dfa_build_start = time.perf_counter()
    dfa = build_dfa(pattern, alphabet)
    dfa_build_time = time.perf_counter() - dfa_build_start
    
    # Pre-build prefix function for KMP
    kmp_build_start = time.perf_counter()
    prefix = compute_prefix_function(pattern)
    kmp_build_time = time.perf_counter() - kmp_build_start
    
    print(f"Preprocessing:")
    print(f"  DFA build time:    {dfa_build_time * 1000:.3f} ms")
    print(f"  KMP prefix time:   {kmp_build_time * 1000:.3f} ms")
    
    # Time DFA search
    dfa_start = time.perf_counter()
    for _ in range(num_runs):
        dfa_search(text, pattern, alphabet)
    dfa_time = (time.perf_counter() - dfa_start) / num_runs
    
    # Time KMP search
    kmp_start = time.perf_counter()
    for _ in range(num_runs):
        kmp_search(text, pattern)
    kmp_time = (time.perf_counter() - kmp_start) / num_runs
    
    print(f"\nSearch time (avg of {num_runs} runs):")
    print(f"  DFA search:        {dfa_time * 1000:.3f} ms")
    print(f"  KMP search:        {kmp_time * 1000:.3f} ms")
    print(f"  Ratio (KMP/DFA):   {kmp_time / dfa_time:.2f}x")
    
    # Verify results match
    dfa_matches = dfa_search(text, pattern, alphabet)
    kmp_matches = kmp_search(text, pattern)
    print(f"\nResults match: {dfa_matches == kmp_matches}")
    print(f"Matches found: {len(dfa_matches)}")


# Run benchmark with DNA sequence
import random
random.seed(42)

dna_alphabet = "ATGC"
dna_text = ''.join(random.choice(dna_alphabet) for _ in range(10000))
dna_pattern = ''.join(random.choice(dna_alphabet) for _ in range(10))

print("DNA Sequence (small alphabet |Σ|=4):")
print("=" * 50)
benchmark_comparison(dna_text, dna_pattern, dna_alphabet)

In [ ]:
# Compare with larger alphabet
ascii_alphabet = ''.join(chr(i) for i in range(32, 127))  # Printable ASCII
ascii_text = ''.join(random.choice(ascii_alphabet) for _ in range(10000))
ascii_pattern = ''.join(random.choice(ascii_alphabet) for _ in range(10))

print("\nASCII Text (large alphabet |Σ|=95):")
print("=" * 50)
benchmark_comparison(ascii_text, ascii_pattern, ascii_alphabet)

### Memory Comparison

In [ ]:
import sys

def estimate_memory(pattern: str, alphabet: str):
    """
    Estimate memory usage for DFA vs KMP.
    """
    m = len(pattern)
    sigma = len(alphabet)
    
    # DFA: dict of dicts with integers
    dfa = build_dfa(pattern, alphabet)
    dfa_size = sys.getsizeof(dfa)
    for state_dict in dfa.values():
        dfa_size += sys.getsizeof(state_dict)
        for key, value in state_dict.items():
            dfa_size += sys.getsizeof(key) + sys.getsizeof(value)
    
    # KMP: just the prefix array
    prefix = compute_prefix_function(pattern)
    kmp_size = sys.getsizeof(prefix)
    
    print(f"Pattern length: {m}, Alphabet size: {sigma}")
    print(f"DFA memory:     ~{dfa_size:,} bytes")
    print(f"KMP memory:     ~{kmp_size:,} bytes")
    print(f"DFA/KMP ratio:  {dfa_size / kmp_size:.1f}x")
    print(f"\nTheoretical:")
    print(f"  DFA:  O(m × |Σ|) = O({m} × {sigma}) = O({m * sigma})")
    print(f"  KMP:  O(m) = O({m})")


print("Memory Usage Comparison")
print("=" * 50)

print("\n1. DNA (small alphabet):")
estimate_memory("ATTCTGATTT", "ATGC")

print("\n2. ASCII (large alphabet):")
estimate_memory("HELLO", ascii_alphabet)

---
## Summary

### Key Takeaways

1. **DFA Concept**: States represent how many characters of the pattern we've matched

2. **Transition Function**: δ(state, char) tells us the longest prefix of P that is a suffix of what we've seen

3. **Construction**: Uses prefix function to efficiently compute fallback transitions in O(m|Σ|)

4. **Search**: Single table lookup per character - O(n) with excellent constant factor

5. **Trade-off**: DFA trades space O(m|Σ|) for simpler per-character operations

### Complexity Summary

| Phase | Time | Space |
|-------|------|-------|
| Build DFA | O(m × \|Σ\|) | O(m × \|Σ\|) |
| Search | O(n) | O(1) |
| **Total** | **O(m × \|Σ\| + n)** | **O(m × \|Σ\|)** |

### When to Use DFA Matching

- ✅ Small, fixed alphabet (DNA, binary, simple character sets)
- ✅ Same pattern searched repeatedly across many texts
- ✅ Performance-critical applications where constant factors matter
- ❌ Large alphabets (Unicode) - use KMP instead
- ❌ Memory-constrained environments - use KMP instead
- ❌ One-time searches with large patterns - preprocessing overhead not amortized

## Alternative Implementation (Kodomo)

This version builds the automaton using a dedicated class and DNA-specific alphabet. The prefix function computes state transitions, and the search method walks the automaton printing each state change.

Key differences from the implementation above:
- A `_DFANode` object stores transitions for each state instead of a flat `dict[int, dict]`.
- The alphabet is fixed to `("A", "T", "G", "C")`, which keeps the transition table small.
- `DFAMatcher` is instantiated once per pattern and can search multiple texts without rebuilding.

In [ ]:
def _prefix_length(pattern: str, probe: str) -> int:
    """Length of the longest prefix of pattern that is a suffix of probe."""
    combined = pattern + "#" + probe + "$"
    sp = [0] * len(combined)
    j = 0
    for i in range(1, len(combined)):
        if combined[i] == "$":
            break
        while j > 0 and combined[i] != combined[j]:
            j = sp[j - 1]
        if combined[i] == combined[j]:
            j += 1
        sp[i] = j
    return sp[len(pattern) + 1 + len(probe) - 1]  # last position before '$'


class _DFANode:
    """Single state in the DFA; transitions keyed by character."""
    def __init__(self, transitions: dict):
        self.dest = transitions


class DFAMatcher:
    """DFA-based exact pattern matcher for a fixed DNA alphabet (A, T, G, C).

    Build once per pattern, then call search() on any number of texts.
    """

    ALPHABET = ("A", "T", "G", "C")

    def __init__(self, pattern: str):
        self.pattern = pattern
        self.nodes = []
        for i in range(len(pattern) + 1):
            prefix = pattern[:i]
            transitions = {
                ch: _prefix_length(pattern, prefix + ch)
                for ch in self.ALPHABET
            }
            self.nodes.append(_DFANode(transitions))

    def search(self, text: str) -> list[int]:
        """Return all 0-based start positions of pattern in text."""
        state = 0
        accept = len(self.nodes) - 1
        matches = []
        for i, ch in enumerate(text):
            if ch not in self.nodes[state].dest:
                continue  # character not in alphabet — stay
            state = self.nodes[state].dest[ch]
            if state == accept:
                matches.append(i - len(self.pattern) + 1)
        return matches


# Example: DNA pattern matching
pattern = "ATTCTGATTT"
text    = "AATGCCGTATTCTATTCTGATTTCTGAATTCTGATTTTTAGT"

dfa = DFAMatcher(pattern)
print(f"Pattern : {pattern!r}")
print(f"Matches : {dfa.search(text)}")

# Reuse the same automaton on a second text
text2 = "ATTCTGATTTATTCTGATTT"
print(f"Text2 matches: {dfa.search(text2)}")

---

## Exercises

### Exercise 1: DFA State Trace for a Repeated Motif (*)

One advantage of a DFA over KMP is that the state at any position is self-contained — you can inspect and display it without any additional bookkeeping. This makes DFAs ideal for teaching and debugging.

**Task:** Build a DFA for the pattern `"ATCGATCG"` using `build_dfa`. Then implement `trace_dfa_states(dfa, pattern, text)` that returns a list of `(position, character, new_state)` tuples showing the state transition at every position. Print a formatted table and mark accepting states (state == `len(pattern)`).

In [ ]:
PATTERN = "ATCGATCG"
DNA_ALPHABET = "ACGT"

TEXT = "GCATCGATCGATCGATCGTTACGATCGATCG"


def trace_dfa_states(
    dfa: dict, pattern: str, text: str
) -> list[tuple[int, str, int]]:
    """
    Trace DFA state transitions for every character in text.

    Args:
        dfa: Transition table from build_dfa(pattern, alphabet)
        pattern: The pattern the DFA was built for
        text: The text to process

    Returns:
        List of (position, character, state_after) for each position in text
    """
    # YOUR CODE HERE
    pass


# dfa = build_dfa(PATTERN, DNA_ALPHABET)
# trace = trace_dfa_states(dfa, PATTERN, TEXT)
#
# accepting = len(PATTERN)
# print(f"Pattern: {PATTERN}  (accepting state = {accepting})")
# print(f"{'pos':>4}  {'char':>4}  {'state':>6}  {'note'}")
# print("-" * 35)
# for pos, char, state in trace:
#     note = "*** MATCH ***" if state == accepting else ""
#     print(f"{pos:>4}  {char:>4}  {state:>6}  {note}")

### Exercise 2: Finding Overlapping Matches with a DFA (**)

Standard pattern search reports a match and resets the automaton. But in genomics, overlapping occurrences matter — for example, counting all occurrences of a tandem repeat unit, even when they overlap.

**Task:** Implement `dfa_search_overlapping(text, pattern, alphabet)` that finds **all** overlapping occurrences of the pattern. After each match, instead of resetting to state 0, use the DFA's own transition to determine the correct state to continue from (i.e., feed the just-accepted character back through the failure/overlap logic).

Hint: after a match at position `i` (state reaches `m`), the next state is `dfa[m][text[i]]` if the DFA has a self-loop for overlap, or equivalently compute it as `dfa[m][text[i]]` — but note that `build_dfa` already populates state `m` transitions using the prefix function, so the automaton naturally handles this if you do not manually reset to 0.

In [ ]:
OVERLAP_PATTERN = "ABA"
OVERLAP_TEXT    = "ABABABA"
# Expected overlapping matches at positions: 0, 2, 4

REPEAT_PATTERN = "ATATAT"
REPEAT_TEXT    = "ATATATATATAT"


def dfa_search_overlapping(text: str, pattern: str, alphabet: str = None) -> list[int]:
    """
    Find all overlapping occurrences of pattern in text using a DFA.

    After each match, do NOT reset the state to 0. Instead let the DFA
    transition table determine the next state — it encodes the correct
    partial match carried over from the suffix of the just-found occurrence.

    Args:
        text: The text to search in
        pattern: The pattern to search for
        alphabet: Character alphabet (defaults to unique chars in text+pattern)

    Returns:
        List of start positions of all overlapping matches
    """
    # YOUR CODE HERE
    pass


# for text, pattern in [(OVERLAP_TEXT, OVERLAP_PATTERN), (REPEAT_TEXT, REPEAT_PATTERN)]:
#     positions = dfa_search_overlapping(text, pattern, DNA_ALPHABET)
#     print(f"'{pattern}' in '{text}': {positions}")

---

[← Previous: Rabin-Karp Algorithm](../../Tier_4_Algorithms_and_Data_Structures/07_String_Algorithms/03_rabin_karp.ipynb) | [Next: Trie (Prefix Tree) →](../../Tier_4_Algorithms_and_Data_Structures/08_Advanced_String_Structures/01_tries.ipynb)